In [4]:
import os
import shutil
import chromadb
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
def indexar_documentos_local(pasta_pdfs="./pdfs", diretorio_banco="./db_chroma"):
    # Limpeza preventiva
    if os.path.exists(diretorio_banco):
        shutil.rmtree(diretorio_banco)
    
    # Carregamento e Split (igual ao anterior)
    documentos = []
    for arquivo in os.listdir(pasta_pdfs):
        if arquivo.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(pasta_pdfs, arquivo))
            documentos.extend(loader.load())

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    chunks = text_splitter.split_documents(documentos)

    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={'device': 'cpu'}
    )

    # --- NOVO JEITO MAIS SEGURO ---
    print("Conectando ao banco...")
    client = chromadb.PersistentClient(path=diretorio_banco)
    
    # Criamos a coleção manualmente antes de enviar os documentos
    collection = client.get_or_create_collection(
        name="concorrencia_db", 
        metadata={"hnsw:space": "cosine"}
    )

    print(f"Vetorizando {len(chunks)} trechos...")
    # Usamos o LangChain para popular esse cliente que já abrimos
    vector_store = Chroma(
        client=client,
        collection_name="concorrencia_db",
        embedding_function=embeddings
    )
    
    # Adicionamos os documentos em blocos para não travar
    vector_store.add_documents(chunks)

    print(f"✅ Sucesso! Banco salvo em: {diretorio_banco}")
    return vector_store

In [6]:
indexar_documentos_local()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Conectando ao banco...
Vetorizando 2223 trechos...
✅ Sucesso! Banco salvo em: ./db_chroma
